In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/uci-secom.csv")
y = df["Pass/Fail"]
feature_cols = [c for c in df.columns if c not in ["Time", "Pass/Fail"]]
X = df[feature_cols]

# Structural clean
missing_pct = X.isnull().mean()
keep = missing_pct[missing_pct <= 0.5].index
X = X[keep]
X = X.drop(columns=X.columns[X.nunique() <= 1])
X = X.fillna(X.median())

# Baseline = mean & std of PASSING wafers (what "normal" looks like)
passing = X[y == -1]
norm_mean = passing.mean()
norm_std = passing.std().replace(0, 1)   # avoid divide-by-zero

# For each FAILING wafer, z-score every sensor vs normal, find top deviations
failing = X[y == 1]
z = (failing - norm_mean) / norm_std

signatures = []
for idx, row in z.iterrows():
    top = row.abs().sort_values(ascending=False).head(5)
    sig = [{"sensor": s, "z": round(row[s], 2)} for s in top.index]
    signatures.append({"wafer_idx": idx, "top_sensors": sig})

print("Built signatures for", len(signatures), "failing wafers")
print("\nExample (wafer", signatures[0]["wafer_idx"], "):")
for s in signatures[0]["top_sensors"]:
    print(f"  Sensor {s['sensor']}: z-score {s['z']}")

Built signatures for 104 failing wafers

Example (wafer 2 ):
  Sensor 567: z-score 6.74
  Sensor 565: z-score 6.62
  Sensor 558: z-score 6.19
  Sensor 569: z-score 5.25
  Sensor 510: z-score 5.21


In [2]:
def build_prompt(signature):
    sensor_lines = "\n".join(
        f"- Sensor {s['sensor']}: {s['z']:+.1f} standard deviations from normal"
        for s in signature["top_sensors"]
    )
    return f"""You are a semiconductor failure-analysis engineer writing a brief FA note for a wafer that failed end-of-line testing.

The most abnormal sensor readings for this wafer were:
{sensor_lines}

Write a concise failure-analysis note (3-4 sentences) that:
- Describes the observed symptom in plain engineering language
- Proposes a plausible root cause (process step, equipment, or material)
- Suggests a corrective action

Vary your wording and the inferred root cause naturally based on which sensors deviated and by how much. Do not use a fixed template. Write only the note, no preamble."""

In [4]:
# import os, json, time
# from dotenv import load_dotenv
# from anthropic import Anthropic

# load_dotenv()  # reads .env
# client = Anthropic()  # picks up ANTHROPIC_API_KEY automatically

# def build_prompt(signature):
#     sensor_lines = "\n".join(
#         f"- Sensor {s['sensor']}: {s['z']:+.1f} standard deviations from normal"
#         for s in signature["top_sensors"]
#     )
#     return f"""You are a semiconductor failure-analysis engineer writing a brief FA note for a wafer that failed end-of-line testing.

# The most abnormal sensor readings for this wafer were:
# {sensor_lines}

# Write a concise failure-analysis note (3-4 sentences) that:
# - Describes the observed symptom in plain engineering language
# - Proposes a plausible root cause (process step, equipment, or material)
# - Suggests a corrective action

# Vary your wording and the inferred root cause naturally based on which sensors deviated and by how much. Do not use a fixed template. Write only the note, no preamble."""

# notes = []
# for i, sig in enumerate(signatures):
#     msg = client.messages.create(
#         model="claude-sonnet-4-6",
#         max_tokens=300,
#         messages=[{"role": "user", "content": build_prompt(sig)}],
#     )
#     note_text = msg.content[0].text.strip()
#     notes.append({
#         "wafer_idx": sig["wafer_idx"],
#         "top_sensors": sig["top_sensors"],
#         "fa_note": note_text,
#     })
#     if (i + 1) % 10 == 0:
#         print(f"Generated {i+1}/{len(signatures)}")
#     time.sleep(0.3)  # gentle on rate limits

# # SAVE so we never regenerate
# with open("../data/processed/fa_notes.json", "w") as f:
#     json.dump(notes, f, indent=2)

# print("Done. Saved", len(notes), "notes.")
# print("\nExample note (wafer", notes[0]["wafer_idx"], "):")
# print(notes[0]["fa_note"])

In [5]:
import json, random
random.seed(42)

# Vocabulary pools — varied so notes aren't identical templates
symptoms = [
    "exhibited parametric drift at final electrical test",
    "showed out-of-spec readings during end-of-line testing",
    "failed functional verification with anomalous measurements",
    "presented elevated leakage indicators at wafer sort",
    "registered abnormal process signatures flagged at test",
]
causes = [
    "deposition thickness non-uniformity in the affected chamber",
    "an etch-rate excursion likely tied to chamber conditioning",
    "a possible calibration drift on the implicated tool",
    "temperature instability during the thermal process step",
    "contamination or particle events in the process module",
]
actions = [
    "Recommend chamber requalification and SPC chart review for the flagged sensors.",
    "Suggest tool recalibration and a follow-up correlation study on adjacent lots.",
    "Advise holding affected lots pending FA confirmation and a maintenance check.",
    "Propose a focused DOE on the implicated process step to confirm root cause.",
    "Initiate cross-lot trend analysis to determine if the excursion is systemic.",
]

def make_note(sig):
    sensors = sig["top_sensors"]
    top = sensors[0]
    n_extreme = sum(1 for s in sensors if abs(s["z"]) > 4)
    sensor_list = ", ".join(f"sensor {s['sensor']} ({s['z']:+.1f}σ)" for s in sensors[:3])
    severity = "severe" if abs(top["z"]) > 5 else "moderate"
    return (
        f"Wafer {sig['wafer_idx']} {random.choice(symptoms)}. "
        f"The {severity} deviation was concentrated in {sensor_list}, "
        f"with {n_extreme} parameters exceeding 4 standard deviations. "
        f"Root cause is suspected to be {random.choice(causes)}. "
        f"{random.choice(actions)}"
    )

notes = [{
    "wafer_idx": sig["wafer_idx"],
    "top_sensors": sig["top_sensors"],
    "fa_note": make_note(sig),
} for sig in signatures]

with open("../data/processed/fa_notes.json", "w") as f:
    json.dump(notes, f, indent=2)

print("Generated", len(notes), "notes (offline).")
print("\nExample:\n", notes[0]["fa_note"])
print("\nAnother:\n", notes[5]["fa_note"])

Generated 104 notes (offline).

Example:
 Wafer 2 exhibited parametric drift at final electrical test. The severe deviation was concentrated in sensor 567 (+6.7σ), sensor 565 (+6.6σ), sensor 558 (+6.2σ), with 5 parameters exceeding 4 standard deviations. Root cause is suspected to be deposition thickness non-uniformity in the affected chamber. Advise holding affected lots pending FA confirmation and a maintenance check.

Another:
 Wafer 38 showed out-of-spec readings during end-of-line testing. The severe deviation was concentrated in sensor 557 (+23.0σ), sensor 551 (+22.4σ), sensor 554 (+22.3σ), with 5 parameters exceeding 4 standard deviations. Root cause is suspected to be contamination or particle events in the process module. Initiate cross-lot trend analysis to determine if the excursion is systemic.


In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

with open("../data/processed/fa_notes.json") as f:
    notes = json.load(f)

texts = [n["fa_note"] for n in notes]

# Load a compact, fast, high-quality embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(texts, show_progress_bar=True)
print("Embeddings shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]